# Recherche parallèle : ratios de paiement et codifications, avec la codification anticipée

*Recherche parallèle, descriptive, **sans modèle**. Journal : [`tableau_resultats.md`](tableau_resultats.md).*

L'EDA du projet (page 3.3 du site, EDA_lab) a croisé les ratios de paiement avec les codifications **selon la convention actuelle** : la codification `PAY_n` rattachée au paiement du même mois, `PAY_AMTn`, sur la facture du mois précédent, `BILL_AMT(n+1)`. Les notebooks `04` à `06` ont montré que la codification suit **le règlement de la facture du mois**, le mois suivant : `PAY_n` rattachée à `PAY_AMT(n-1)` sur `BILL_AMTn` (**codification anticipée**). La paire paiement / facture est la même (un paiement règle la facture du mois précédent) ; seule la codification rattachée change d'un mois.

Ce notebook refait les croisements de l'EDA avec la codification anticipée, à côté de la convention actuelle :
1. pour chaque codification, la répartition des ratios de paiement ;
2. pour chaque tranche de ratio, la part des retards et des codifications saines ;
3. les sorties du retard selon le ratio de paiement (l'analyse « mois du paiement / mois suivant » de la page 3.3) ;
4. la stabilité mois par mois.

**Ratio de paiement** : paiement / facture qu'il règle × 100 (définition de `ratio_PAY_BILLn`, `docs/colonnes_creees.md`), en tranches comme sur la page 3.3 : 0 %, de 0 à 3 %, 3 à 5 %, 5 à 10 %, 10 à 20 %, 20 à 50 %, 50 à 90 %, 90 % et plus ; plus « aucune facture due ». **Mois comparés** : mai à août (`PAY_5` à `PAY_2`), seuls mois où les deux conventions se calculent. Niveau 0, sans `dpnm`, sans les clients du test du ML de septembre ; codifications 1 écartées (rares avant septembre).

## 1. Données

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv").drop(columns="dpnm")   # dpnm jamais utilisé
connus = df[df["PAY_1"] != 1]
_, test_ml = train_test_split(connus.index, test_size=0.2, stratify=connus["PAY_1"], random_state=42)
df = df.drop(index=test_ml)

TRANCHES = ["aucune facture due", "0 %", "0 à 3 %", "3 à 5 %", "5 à 10 %", "10 à 20 %", "20 à 50 %", "50 à 90 %", "90 % et plus"]
def tranche(paiement, facture):
    r = paiement / facture.where(facture > 0) * 100
    return pd.Series(np.select([facture <= 0, paiement <= 0, r < 3, r < 5, r < 10, r < 20, r < 50, r < 90], TRANCHES[:8], TRANCHES[8]), index=paiement.index)

def codif_lisible(c):
    return c.astype(int).astype(str).where(c < 3, "3 et plus")

MOIS = {2: "août", 3: "juillet", 4: "juin", 5: "mai"}
lignes = []
for n in range(2, 6):
    lignes.append(pd.DataFrame({
        "client": df.index, "mois": n, "codif": df[f"PAY_{n}"], "codif_avant": df[f"PAY_{n + 1}"],
        "tranche_A": tranche(df[f"PAY_AMT{n}"], df[f"BILL_AMT{n + 1}"]),          # convention actuelle
        "tranche_B": tranche(df[f"PAY_AMT{n - 1}"], df[f"BILL_AMT{n}"]),          # codification anticipée
    }, index=df.index))
ex = pd.concat(lignes, ignore_index=True)
ex = ex[(ex["codif"] != 1) & (ex["codif_avant"] != 1)].reset_index(drop=True)
ex["codification"] = codif_lisible(ex["codif"])
ORDRE_CODIF = ["-2", "-1", "0", "2", "3 et plus"]
print(f"Exemples (client × mois, mai à août) : {len(ex)} ; clients : {ex['client'].nunique()}")

Exemples (client × mois, mai à août) : 98914 ; clients : 24737


## 2. Pour chaque codification, la répartition des ratios de paiement

Deux tableaux, un par convention : une ligne par codification, la répartition de ses exemples entre les tranches de ratio (chaque ligne fait 100 %).

In [7]:
for convention, libelle in [("A", "Convention actuelle : paiement du mois sur la facture du mois précédent"),
                            ("B", "Codification anticipée : facture du mois, réglée le mois suivant")]:
    t = pd.crosstab(ex["codification"], ex[f"tranche_{convention}"], normalize="index").reindex(index=ORDRE_CODIF, columns=TRANCHES).fillna(0) * 100
    t.insert(0, "Exemples", ex["codification"].value_counts().reindex(ORDRE_CODIF))
    display(Markdown(f"**{libelle}** : en % de chaque codification"))
    display(t.round(1))
    n = pd.crosstab(ex["codification"], ex[f"tranche_{convention}"], margins=True, margins_name="Total").reindex(index=ORDRE_CODIF + ["Total"], columns=TRANCHES + ["Total"]).fillna(0).astype(int)
    display(Markdown(f"**{libelle}** : en nombre d'exemples (client × mois)"))
    display(n)

**Convention actuelle : paiement du mois sur la facture du mois précédent** : en % de chaque codification

tranche_A,Exemples,aucune facture due,0 %,0 à 3 %,3 à 5 %,5 à 10 %,10 à 20 %,20 à 50 %,50 à 90 %,90 % et plus
codification,,,,,,,,,,
-2,14221,57.6,0.2,0.3,0.4,0.2,0.2,0.2,0.3,40.6
-1,19166,16.1,7.5,2.2,2.3,1.4,1.1,2.1,2.8,64.6
0,52530,4.1,4.3,5.3,41.7,20.9,10.0,6.4,1.9,5.4
2,11697,3.9,24.2,4.4,28.0,19.2,8.4,3.3,1.3,7.4
3 et plus,1300,0.0,70.8,5.8,11.2,8.7,2.7,0.3,0.3,0.2


**Convention actuelle : paiement du mois sur la facture du mois précédent** : en nombre d'exemples (client × mois)

tranche_A,aucune facture due,0 %,0 à 3 %,3 à 5 %,5 à 10 %,10 à 20 %,20 à 50 %,50 à 90 %,90 % et plus,Total
codification,,,,,,,,,,
-2,8190,28,42,52,32,27,35,46,5769,14221
-1,3078,1431,426,435,261,218,396,536,12385,19166
0,2164,2266,2777,21927,10976,5233,3340,1005,2842,52530
2,459,2827,519,3272,2242,978,381,157,862,11697
3 et plus,0,921,76,145,113,35,4,4,2,1300
Total,13891,7473,3840,25831,13624,6491,4156,1748,21860,98914


**Codification anticipée : facture du mois, réglée le mois suivant** : en % de chaque codification

tranche_B,Exemples,aucune facture due,0 %,0 à 3 %,3 à 5 %,5 à 10 %,10 à 20 %,20 à 50 %,50 à 90 %,90 % et plus
codification,,,,,,,,,,
-2,14221,65.1,0.3,0.3,0.5,0.3,0.2,0.3,0.3,32.7
-1,19166,10.6,0.2,0.1,0.1,0.1,0.1,0.3,0.4,88.1
0,52530,2.0,2.2,4.8,42.2,24.8,12.1,7.9,2.9,1.1
2,11697,2.5,46.1,7.2,17.8,15.1,7.6,2.5,0.9,0.4
3 et plus,1300,0.0,80.3,5.8,5.4,6.2,1.5,0.5,0.2,0.2


**Codification anticipée : facture du mois, réglée le mois suivant** : en nombre d'exemples (client × mois)

tranche_B,aucune facture due,0 %,0 à 3 %,3 à 5 %,5 à 10 %,10 à 20 %,20 à 50 %,50 à 90 %,90 % et plus,Total
codification,,,,,,,,,,
-2,9263,37,40,66,43,32,45,41,4654,14221
-1,2037,38,11,10,18,26,61,86,16879,19166
0,1058,1155,2503,22171,13018,6370,4164,1530,561,52530
2,292,5392,838,2077,1764,889,294,102,49,11697
3 et plus,0,1044,75,70,81,20,6,2,2,1300
Total,12650,7666,3467,24394,14924,7337,4570,1761,22145,98914


**Comment lire** : sous une convention qui colle, chaque codification a un paiement typique : -2 sans facture due ou facture soldée (90 % et plus), -1 facture soldée, 0 paiement partiel au-dessus de la mensualité, retards (2 et plus) rien payé ou moins de 3 %. Sous une convention qui ne colle pas, les codifications se mélangent dans toutes les tranches.

## 3. Pour chaque tranche de ratio, la part des retards et des codifications saines

Une ligne par tranche de ratio : la répartition des codifications (chaque ligne fait 100 %), sous chaque convention. La colonne « retard (2 et plus) » répond à la question de la page 3.3 : dans quelle part des mois où le client paie tant la banque pose-t-elle un retard ?

In [8]:
lignes = []
for convention in ("A", "B"):
    t = pd.crosstab(ex[f"tranche_{convention}"], ex["codification"], normalize="index").reindex(index=TRANCHES, columns=ORDRE_CODIF).fillna(0) * 100
    t["retard (2 et plus)"] = t["2"] + t["3 et plus"]
    t.insert(0, "Exemples", ex[f"tranche_{convention}"].value_counts().reindex(TRANCHES).fillna(0).astype(int))
    t.insert(0, "Convention", convention)
    lignes.append(t.round(1).reset_index().rename(columns={f"tranche_{convention}": "Tranche"}))
tableau = pd.concat(lignes)
tableau["Tranche"] = pd.Categorical(tableau["Tranche"], TRANCHES, ordered=True)
display(tableau.sort_values(["Tranche", "Convention"]).set_index(["Tranche", "Convention"]))
for convention in ("A", "B"):
    n = pd.crosstab(ex[f"tranche_{convention}"], ex["codification"], margins=True, margins_name="Total").reindex(index=TRANCHES + ["Total"], columns=ORDRE_CODIF + ["Total"]).fillna(0).astype(int)
    display(Markdown(f"**Convention {convention}** : en nombre d'exemples (client × mois)"))
    display(n)

codification                   Exemples    -2    -1     0     2  3 et plus  \
Tranche            Convention                                                
aucune facture due A              13891  59.0  22.2  15.6   3.3        0.0   
                   B              12650  73.2  16.1   8.4   2.3        0.0   
0 %                A               7473   0.4  19.1  30.3  37.8       12.3   
                   B               7666   0.5   0.5  15.1  70.3       13.6   
0 à 3 %            A               3840   1.1  11.1  72.3  13.5        2.0   
                   B               3467   1.2   0.3  72.2  24.2        2.2   
3 à 5 %            A              25831   0.2   1.7  84.9  12.7        0.6   
                   B              24394   0.3   0.0  90.9   8.5        0.3   
5 à 10 %           A              13624   0.2   1.9  80.6  16.5        0.8   
                   B              14924   0.3   0.1  87.2  11.8        0.5   
10 à 20 %          A               6491   0.4   3.4  80.6  15.1        0.5   
                   B               7337   0.4   0.4  86.8  12.1        0.3   
20 à 50 %          A               4156   0.8   9.5  80.4   9.2        0.1   
                   B               4570   1.0   1.3  91.1   6.4        0.1   
50 à 90 %          A               1748   2.6  30.7  57.5   9.0        0.2   
                   B               1761   2.3   4.9  86.9   5.8        0.1   
90 % et plus       A              21860  26.4  56.7  13.0   3.9        0.0   
                   B              22145  21.0  76.2   2.5   0.2        0.0   

codification                   retard (2 et plus)  
Tranche            Convention                      
aucune facture due A                          3.3  
                   B                          2.3  
0 %                A                         50.2  
                   B                         84.0  
0 à 3 %            A                         15.5  
                   B                         26.3  
3 à 5 %            A                         13.2  
                   B                          8.8  
5 à 10 %           A                         17.3  
                   B                         12.4  
10 à 20 %          A                         15.6  
                   B                         12.4  
20 à 50 %          A                          9.3  
                   B                          6.6  
50 à 90 %          A                          9.2  
                   B                          5.9  
90 % et plus       A                          4.0  
                   B                          0.2

**Convention A** : en nombre d'exemples (client × mois)

codification,-2,-1,0,2,3 et plus,Total
tranche_A,,,,,,
aucune facture due,8190,3078,2164,459,0,13891
0 %,28,1431,2266,2827,921,7473
0 à 3 %,42,426,2777,519,76,3840
3 à 5 %,52,435,21927,3272,145,25831
5 à 10 %,32,261,10976,2242,113,13624
10 à 20 %,27,218,5233,978,35,6491
20 à 50 %,35,396,3340,381,4,4156
50 à 90 %,46,536,1005,157,4,1748
90 % et plus,5769,12385,2842,862,2,21860


**Convention B** : en nombre d'exemples (client × mois)

codification,-2,-1,0,2,3 et plus,Total
tranche_B,,,,,,
aucune facture due,9263,2037,1058,292,0,12650
0 %,37,38,1155,5392,1044,7666
0 à 3 %,40,11,2503,838,75,3467
3 à 5 %,66,10,22171,2077,70,24394
5 à 10 %,43,18,13018,1764,81,14924
10 à 20 %,32,26,6370,889,20,7337
20 à 50 %,45,61,4164,294,6,4570
50 à 90 %,41,86,1530,102,2,1761
90 % et plus,4654,16879,561,49,2,22145


**Comment lire** : deux lignes par tranche, A puis B. Sous la codification anticipée, la part des retards devrait être très forte quand rien n'est payé ou moins de 3 %, et presque nulle dès que la mensualité est payée ; sous la convention actuelle, plus étalée.

## 4. Les sorties du retard selon le ratio de paiement

Les clients en retard le mois précédent (`PAY_(n+1)` >= 2) : part de ceux qui sortent du retard en M (`PAY_n` <= 0), selon la tranche de ratio de paiement, sous chaque convention. C'est l'analyse de la page 3.3 (« la sortie du retard se lit le mois suivant le paiement ») : sous la codification anticipée, le paiement « du mois suivant » de la page devient le paiement rattaché à la codification.

In [9]:
r = ex[ex["codif_avant"] >= 2]
sortie = r["codif"] <= 0
lignes = []
for t in TRANCHES:
    ligne = {"Tranche": t}
    for convention in ("A", "B"):
        m = r[f"tranche_{convention}"] == t
        ligne[f"{convention} : retards"] = int(m.sum())
        ligne[f"{convention} : sortent (%)"] = round(sortie[m].mean() * 100, 1) if m.any() else np.nan
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index("Tranche"))
print(f"Clients en retard le mois précédent : {len(r)} ; sortent en M : {sortie.mean() * 100:.1f} %")

,A : retards,A : sortent (%),B : retards,B : sortent (%)
Tranche,,,,
aucune facture due,264,0.0,2,50.0
0 %,5798,38.2,3114,0.2
0 à 3 %,776,43.9,368,1.9
3 à 5 %,2129,18.3,2042,2.0
5 à 10 %,1650,12.3,2815,38.7
10 à 20 %,706,7.4,1738,50.5
20 à 50 %,227,7.9,683,58.1
50 à 90 %,93,15.1,192,52.6
90 % et plus,49,20.4,738,97.8


Clients en retard le mois précédent : 11692 ; sortent en M : 27.7 %


**Comment lire** : une ligne par tranche ; pour chaque convention, le nombre de retards et la part qui sort en M. Une convention qui colle donne une frontière nette : presque aucune sortie sans paiement, presque toutes au-dessus d'un seuil de paiement.

## 5. Mois par mois

La part des retards dans deux tranches clés (« 0 % » et « 90 % et plus ») et la part de sorties pour un retard payé à 90 % et plus, pour chaque mois M, sous chaque convention.

In [10]:
lignes = []
for n, nom in MOIS.items():
    e = ex[ex["mois"] == n]
    rr = e[e["codif_avant"] >= 2]
    ligne = {"Mois M": nom}
    for convention in ("A", "B"):
        tr = e[f"tranche_{convention}"]
        ligne[f"{convention} : retards si rien payé (%)"] = round((e.loc[tr == "0 %", "codif"] >= 2).mean() * 100, 1)
        ligne[f"{convention} : retards si 90 % et plus (%)"] = round((e.loc[tr == "90 % et plus", "codif"] >= 2).mean() * 100, 1)
        ligne[f"{convention} : sorties si retard payé à 90 % et plus (%)"] = round((rr.loc[rr[f"tranche_{convention}"] == "90 % et plus", "codif"] <= 0).mean() * 100, 1)
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index("Mois M"))

,A : retards si rien payé (%),A : retards si 90 % et plus (%),A : sorties si retard payé à 90 % et plus (%),B : retards si rien payé (%),B : retards si 90 % et plus (%),B : sorties si retard payé à 90 % et plus (%)
Mois M,,,,,,
août,59.8,5.1,35.3,99.0,0.2,98.4
juillet,55.3,5.5,20.0,98.9,0.3,97.8
juin,44.9,3.3,0.0,77.4,0.3,94.1
mai,41.4,1.8,10.0,62.7,0.1,99.3


**Comment lire** : une ligne par mois ; les écarts entre A et B doivent avoir le même sens tous les mois.

## 6. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*